# Faz 1.1 — Kural Tabanlı Motorlu Baseline ÜRETİCİ

**LLM yok.** Parametrik, deterministik motor. Topoloji: **dikdörtgen/kare bina**
içinde **3 oda + 1 koridor + 1 salon**.

```
+-----------------------------+  üst bant: 3 oda (yan yana)
|  Oda1  |  Oda2  |  Oda3     |
+--------+--------+-----------+
|        KORİDOR (tam en)     |  orta bant: yatay koridor
+-----------------------------+
|        SALON (tam en)       |  alt bant: salon
+-----------------------------+
```

**Parametreler** (`BaselineConfig`):

| Parametre | Açıklama | Varsayılan |
|-----------|----------|------------|
| `doors_per_room` | her oda/salonda kaç kapı | 3 |
| `room_min` / `room_max` | oda kenar boyutları (m) | 3 / 5 |
| `corridor_min` / `corridor_max` | koridor genişlik+uzunluk (m) | 3 / 5 |
| `shape` | `rectangle` veya `square` | rectangle |
| `n_baselines` | kaç farklı baseline üretilecek | 1 |

> Ek varsayılan (parametre değil): her oda/salona R2'yi (pencere/taban ≥%10)
> sağlayacak pencere eklenir; koridor sirkülasyon olduğu için R2'den muaftır.
> Mantık `src/ifc_gen/baseline/procedural.py` içinde; motor `layout_to_ifc`'i yeniden kullanır.

## 0) Kurulum

In [ ]:
import sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = REPO / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
print('src:', SRC)

## 1) Konfigürasyon
Senin belirlediğin 5 parametre. (Burada varsayılanlar; istediğini değiştir.)

In [ ]:
from ifc_gen.baseline.procedural import BaselineConfig, generate_baselines

cfg = BaselineConfig(
    doors_per_room = 3,
    room_min = 3.0, room_max = 5.0,
    corridor_min = 3.0, corridor_max = 5.0,
    shape = 'rectangle',
    n_baselines = 1,
    seed = 42,            # tekrar üretilebilirlik için (None = rastgele)
)
print(cfg)

## 2) Üret
`generate_baselines(cfg)` → her varyantı `data/baseline_ifc/` altına IFC + `meta.json` yazar.

In [ ]:
sonuc = generate_baselines(cfg)
print(f'{len(sonuc)} baseline üretildi:')
for r in sonuc:
    pr = r['meta']['params']
    print(f"  {r['ifc_path'].name}")
    print(f"     shape={pr['shape']} W={pr['width']} D={pr['depth']} "
          f"oda_gen={pr['room_widths']} koridor={pr['corridor']} salon={pr['salon_depth']}")
    print(f"     sayılar={r['meta']['counts']}")

## 3) Doğrulama — geçerli mi + kurallara uygun mu?
Her baseline: şema 0 hata olmalı ve **0 kural ihlali** (kural tabanlı baseline temizdir).

In [ ]:
import ifcopenshell
from ifcopenshell import validate
from ifc_gen.inject import rules
from viewer.model import load_viewer_model

for r in sonuc:
    p = str(r['ifc_path'])
    f = ifcopenshell.open(p)
    lg = validate.json_logger(); validate.validate(f, lg)
    vm = load_viewer_model(p)
    viol = rules.violations_only(vm)
    spaces = [e.name for e in vm.elements.values() if e.ifc_type=='IfcSpace']
    print(f"{r['ifc_path'].name}: şema_hata={len(lg.statements)} | "
          f"kural_ihlali={len(viol)} | mekanlar={spaces}")
    for v in viol: print('   ihlal:', v.rule, vm.elements[v.ekey].name, v.detail)

## 4) Görselleştir — ilk baseline

In [ ]:
import matplotlib.pyplot as plt
from viewer import static
vm = load_viewer_model(str(sonuc[0]['ifc_path']))
fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d'); static.plot_3d(vm, ax=ax1)
ax2 = fig.add_subplot(122); static.plot_graph(vm, ax=ax2)
plt.tight_layout(); plt.show()

In [ ]:
from viewer import view
view(str(sonuc[0]['ifc_path']))

## 5) Parametreyi değiştir — çok varyant + kare bina
Örnek: kare bina, oda başına 2 kapı, 3 farklı baseline.

In [ ]:
cfg2 = BaselineConfig(shape='square', doors_per_room=2, n_baselines=3, seed=7)
sonuc2 = generate_baselines(cfg2)
for r in sonuc2:
    pr = r['meta']['params']
    print(f"{r['ifc_path'].name}: shape={pr['shape']} W={pr['width']} D={pr['depth']} "
          f"sayılar={r['meta']['counts']}")

## Özet
- `BaselineConfig` + `generate_baselines()` → 5 parametreli, çok varyantlı **kural tabanlı** baseline.
- Topoloji: 3 oda + koridor + salon; dikdörtgen/kare; her varyant farklı (seed ile tekrar üretilebilir).
- Çıktı: geçerli IFC4 (şema 0 hata) + `meta.json`, **0 kural ihlali**.
- `.py` modül olduğu için diğer fazlardan da çağrılır (ihlal ekleme bu baseline'lar üzerine işler).